In [1]:
import pandas as pd
import numpy as np

In [3]:
#loading csv dataset

df_riverFlow = pd.read_csv("Datasets/melamchi_waterflow.csv")
df_melamchiWeather = pd.read_csv("Datasets/melamchi_weather.csv")

In [4]:
# 2. CREATE A MATCHING DATE COLUMN IN WEATHER DATA
# We combine YEAR and DOY. The '%Y-%j' format tells pandas '%j' means Day of Year (1-365)
df_melamchiWeather['Dates'] = pd.to_datetime(
    df_melamchiWeather['YEAR'].astype(str) + '-' + df_melamchiWeather['DOY'].astype(str), 
    format='%Y-%j'
)

# 3. CONVERT BOTH DATE COLUMNS TO DATETIME OBJECTS
# This ignores formatting differences (like '00:00:00' timestamps) so they match flawlessly
df_riverFlow['Dates'] = pd.to_datetime(df_riverFlow['Dates'])
df_melamchiWeather['Dates'] = pd.to_datetime(df_melamchiWeather['Dates'])

#clean -999 values
df_melamchiWeather = df_melamchiWeather.replace(-999,np.nan).dropna()

In [5]:
#Merging datasets
df_melamchi = pd.merge(df_riverFlow,df_melamchiWeather,on='Dates')
df_melamchi.head()

,Dates,Values,YEAR,DOY,T2M,PRECTOTCORR,RH2M
0,1981-01-01,0.120,1981,1,1.92,0.00,47.48
1,1981-01-02,0.120,1981,2,1.83,0.00,52.88
2,1981-01-03,0.120,1981,3,0.86,0.00,67.05
3,1981-01-04,0.120,1981,4,-0.66,0.02,75.51
4,1981-01-05,0.127,1981,5,-1.09,0.21,72.27


In [6]:
df_melamchi.rename(columns={
    'PRECTOTCORR': 'Rainfall', 
    'T2M': 'Temperature',
    'Values': 'Waterflow',
    'RH2M': 'Relative_Humidity'
},inplace=True)
df_melamchi.head()

,Dates,Waterflow,YEAR,DOY,Temperature,Rainfall,Relative_Humidity
0,1981-01-01,0.120,1981,1,1.92,0.00,47.48
1,1981-01-02,0.120,1981,2,1.83,0.00,52.88
2,1981-01-03,0.120,1981,3,0.86,0.00,67.05
3,1981-01-04,0.120,1981,4,-0.66,0.02,75.51
4,1981-01-05,0.127,1981,5,-1.09,0.21,72.27


In [7]:
#making the model

import torch 
import torch.nn as nn
import torch.optim as optim

class floodPredictionModel(nn.Module):
    def __init__(self,input_size):
        super().__init__()
        self.linear1 = nn.Linear(input_size,64)
        self.hidden = nn.Linear(64,64)
        self.relu = nn.ReLU()
        self.linear2 = nn.Linear(64,1)
        
    def forward(self,x):
        x = self.linear1(x)
        x = self.relu(x)
        x = self.hidden(x)
        x = self.relu(x)
        x = self.linear2(x)
        return x

In [8]:
#Creating flood occurrence label
flood_threshold = df_melamchi['Waterflow'].quantile(0.95)
df_melamchi['floodOccurrence'] = (df_melamchi['Waterflow'] > flood_threshold).astype(int)
df_melamchi.head()

,Dates,Waterflow,YEAR,DOY,Temperature,Rainfall,Relative_Humidity,floodOccurrence
0,1981-01-01,0.120,1981,1,1.92,0.00,47.48,0
1,1981-01-02,0.120,1981,2,1.83,0.00,52.88,0
2,1981-01-03,0.120,1981,3,0.86,0.00,67.05,0
3,1981-01-04,0.120,1981,4,-0.66,0.02,75.51,0
4,1981-01-05,0.127,1981,5,-1.09,0.21,72.27,0


In [9]:
#making prediction dataset to calculate loss
y = df_melamchi.pop('floodOccurrence')
X_raw = df_melamchi.drop(columns=['Dates','YEAR','DOY'])

#splitting 80% for training and 20% for validation
split_idx = int(len(X_raw) * 0.8)
X_train_raw = X_raw.iloc[:split_idx]
y_train_raw = y.iloc[:split_idx]

X_val_raw = X_raw.iloc[split_idx:]
y_val_raw = y.iloc[split_idx:]
X_raw.head()

,Waterflow,Temperature,Rainfall,Relative_Humidity
0,0.120,1.92,0.00,47.48
1,0.120,1.83,0.00,52.88
2,0.120,0.86,0.00,67.05
3,0.120,-0.66,0.02,75.51
4,0.127,-1.09,0.21,72.27


In [10]:
#Scaling the data for good training
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_raw)
X_val_scaled = scaler.transform(X_val_raw)

# Convert everything to PyTorch Tensors (Float32)
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train_raw.values, dtype=torch.float32).unsqueeze(1) # shape: [N, 1]

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)
y_val_tensor = torch.tensor(y_val_raw.values, dtype=torch.float32).unsqueeze(1)

In [11]:
#training model
flood_model = floodPredictionModel(input_size=4)
criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(flood_model.parameters(),lr=0.001)
epochs = 200

for epoch in range(epochs):
    
    flood_model.train()
    optimizer.zero_grad()
    
    train_predictions = flood_model(X_train_tensor)
    train_loss = criterion(train_predictions, y_train_tensor)
    
    train_loss.backward()
    optimizer.step()
    
    # --- VALIDATION PHASE ---
    flood_model.eval() # Switch off layers like dropout/batchnorm
    with torch.no_grad(): # Turn off gradient computation to save memory/speed
        val_predictions = flood_model(X_val_tensor)
        val_loss = criterion(val_predictions, y_val_tensor)
        
    # Print metrics every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"Epoch [{epoch+1}/{epochs}] -> Train Loss: {train_loss.item():.4f} | Val Loss: {val_loss.item():.4f}")
    
torch.save(flood_model.state_dict(), 'flood_prediction_weights.pth')#saving the weights

Epoch [1/200] -> Train Loss: 0.7177 | Val Loss: 0.7018
Epoch [5/200] -> Train Loss: 0.6455 | Val Loss: 0.6305
Epoch [10/200] -> Train Loss: 0.5624 | Val Loss: 0.5503
Epoch [15/200] -> Train Loss: 0.4872 | Val Loss: 0.4771
Epoch [20/200] -> Train Loss: 0.4189 | Val Loss: 0.4102
Epoch [25/200] -> Train Loss: 0.3583 | Val Loss: 0.3510
Epoch [30/200] -> Train Loss: 0.3054 | Val Loss: 0.2994
Epoch [35/200] -> Train Loss: 0.2591 | Val Loss: 0.2548
Epoch [40/200] -> Train Loss: 0.2196 | Val Loss: 0.2171
Epoch [45/200] -> Train Loss: 0.1872 | Val Loss: 0.1864
Epoch [50/200] -> Train Loss: 0.1616 | Val Loss: 0.1621
Epoch [55/200] -> Train Loss: 0.1415 | Val Loss: 0.1429
Epoch [60/200] -> Train Loss: 0.1254 | Val Loss: 0.1278
Epoch [65/200] -> Train Loss: 0.1126 | Val Loss: 0.1158
Epoch [70/200] -> Train Loss: 0.1021 | Val Loss: 0.1059
Epoch [75/200] -> Train Loss: 0.0931 | Val Loss: 0.0972
Epoch [80/200] -> Train Loss: 0.0852 | Val Loss: 0.0896
Epoch [85/200] -> Train Loss: 0.0783 | Val Loss: 0

In [13]:
from sklearn.metrics import classification_report, confusion_matrix

# 1. Switch model to evaluation mode
flood_model.eval()

# 2. Get raw logits from the validation set without tracking gradients
with torch.no_grad():
    val_logits = flood_model(X_val_tensor)
    # Pass logits through Sigmoid to get probability percentages (0.0 to 1.0)
    val_probs = torch.sigmoid(val_logits)
    
    # If probability is >= 0.5, predict a flood (1), otherwise no flood (0)
    val_preds = (val_probs >= 0.5).int().numpy()

# 3. Convert true validation tensor back to a numpy array for scikit-learn
y_true = y_val_tensor.numpy()

# 4. Print out the performance breakdown
print("--- Confusion Matrix ---")
print(confusion_matrix(y_true, val_preds))

print("\n--- Classification Report ---")
print(classification_report(y_true, val_preds, target_names=['No Flood', 'Flood']))

--- Confusion Matrix ---
[[3149    0]
 [  21  118]]

--- Classification Report ---
              precision    recall  f1-score   support

    No Flood       0.99      1.00      1.00      3149
       Flood       1.00      0.85      0.92       139

    accuracy                           0.99      3288
   macro avg       1.00      0.92      0.96      3288
weighted avg       0.99      0.99      0.99      3288

